# IELTS Listening Audio Pipeline — walkthrough

This notebook shows what `run.py` does under the hood, one step at a time: parsing a transcript, assigning voices, and calling Google Cloud Text-to-Speech.

**Requirements:** a working `.env` with `GOOGLE_APPLICATION_CREDENTIALS` set (see the main README's Step 3) — the synthesis cells make real, low-cost calls to your Google Cloud project.

In [1]:
import sys
sys.path.insert(0, "..")

from IPython.display import Audio, display

from src.parser import parse_script
from src.voices import assign_dialogue_voices, assign_narrator_voice
from src.tts_client import get_client, synthesize

## 1. Parsing a transcript

Input files follow a fixed format: an optional `# GENDER:` header block, then `Speaker: line` turns.

In [2]:
part1_text = """# GENDER: Examiner=male
# GENDER: Candidate=female

Examiner: Good morning. Can you tell me your full name, please?
Candidate: Yes, my name is Sarah Thompson.
Examiner: And where are you from, Sarah?
Candidate: I'm originally from Manchester, in the north of England.
"""

script = parse_script(part1_text, part=1)

print("Speakers:", script.speakers)
print("Genders:", script.genders)
print("Is dialogue:", script.is_dialogue)
print()
for turn in script.turns:
    print(f"{turn.speaker} ({turn.gender}): {turn.text}")

Speakers: ['Examiner', 'Candidate']
Genders: {'Examiner': 'male', 'Candidate': 'female'}
Is dialogue: True

Examiner (male): Good morning. Can you tell me your full name, please?
Candidate (female): Yes, my name is Sarah Thompson.
Examiner (male): And where are you from, Sarah?
Candidate (female): I'm originally from Manchester, in the north of England.


## 2. Voice assignment

Each speaker is matched to the least-used voice in their declared gender's pool. Parts 1–2 stay British English (`en-GB`) only; Parts 3–4 also draw from Australian, Indian, and American English, mirroring how the real exam sometimes varies accents in its harder sections.

In [3]:
voice_map = assign_dialogue_voices(script.speakers, script.genders, part=script.part)
for speaker, (voice_name, language_code) in voice_map.items():
    print(f"{speaker} -> {voice_name} ({language_code})")

Examiner -> en-GB-Neural2-B (en-GB)
Candidate -> en-GB-Neural2-A (en-GB)


## 3. Synthesizing one turn (real API call)

This calls your actual Google Cloud project — a few sentences costs a negligible fraction of the free monthly allowance.

In [4]:
client = get_client()

first_turn = script.turns[0]
voice_name, language_code = voice_map[first_turn.speaker]
audio_bytes = synthesize(client, first_turn.text, voice_name, language_code=language_code)

print(f"{len(audio_bytes)} bytes synthesized using {voice_name} ({language_code})")
display(Audio(audio_bytes))

26688 bytes synthesized using en-GB-Neural2-B (en-GB)


## 4. Monologue example (Part 2)

Parts 2 and 4 are usually a single narrator, not a dialogue — no speaker labels are read aloud, only the spoken text.

In [5]:
part2_text = """# GENDER: Narrator=female

Narrator: Welcome to Part 2. In this section you will hear a talk about the opening hours of the new city library.
"""

part2_script = parse_script(part2_text, part=2)
narrator_gender = part2_script.genders.get(part2_script.speakers[0])
voice_name, language_code = assign_narrator_voice(narrator_gender, part=part2_script.part)

full_text = " ".join(turn.text for turn in part2_script.turns)
audio_bytes = synthesize(client, full_text, voice_name, language_code=language_code)

print(f"{len(audio_bytes)} bytes synthesized using {voice_name} ({language_code})")
display(Audio(audio_bytes))

52992 bytes synthesized using en-GB-Neural2-C (en-GB)


## 5. Accent variation in Parts 3/4

Gender matching always wins first — a speaker declared `male`/`female` only ever gets a voice of that gender. Accent is layered on top: once a part's UK pool for a gender is used up within the same script, later speakers of that gender spill into the wider Australian/Indian/American pool rather than repeating a voice.

In [6]:
speakers = [f"Student{i}" for i in range(4)]
genders = {s: "male" for s in speakers}

part3_map = assign_dialogue_voices(speakers, genders, part=3)
for speaker, (voice_name, language_code) in part3_map.items():
    print(f"{speaker} -> {voice_name} ({language_code})")

Student0 -> en-GB-Neural2-D (en-GB)
Student1 -> en-AU-Neural2-B (en-AU)
Student2 -> en-AU-Neural2-D (en-AU)
Student3 -> en-IN-Neural2-B (en-IN)


## Running the full pipeline

For a complete 4-part test, put `part1.txt`–`part4.txt` in `transcripts/` and run from the project root:

```bash
python run.py
```

Output is saved to `output/test1/`, `output/test2/`, and so on.